## Churn Indicators:
### Goal:
 Build a customer activity profile to help marketing identify at-risk customers (no predictions).
### Why it matters: 
Allows timely retention actions by analysts.
### How to do it:
- For each customer_id, compute:
- • Days since last order
- • Average gap between orders
- • % change in spend over last N periods
- Tag customers based on inactivity thresholds (e.g.,
->45 days = “at risk”)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType

In [0]:
df_fact_order = spark.read.table('global_partner_project.gold.fact_order')

In [0]:
df_agg_fact_order = df_fact_order.groupBy('creation_time_utc','user_id','order_id').agg(F.sum('item_price').alias('amount'))

In [0]:
print(df_agg_fact_order.count())

In [0]:
df_agg_fact_order1 = df_agg_fact_order.groupBy('user_id').agg(F.count('user_id').alias('No_of_orders')).orderBy('No_of_orders')
#df_agg_fact_order1.display()

In [0]:
df_one_order_users = df_agg_fact_order1.filter(F.col('No_of_orders')==1).select('user_id')
print(df_one_order_users.count())
df_multiple_order_users = df_agg_fact_order1.filter(F.col('No_of_orders')>1).select('user_id')
print(df_multiple_order_users.count())

### Create the below df for one time users to unionByName later

In [0]:
df_onetime_user_gap = df_one_order_users.withColumn('average_gap',F.lit('OneTimeUser'))
df_onetime_user_perct = df_one_order_users.withColumn('avg_perct_chg',F.lit('OneTimeUser'))
                                        

### Filter for user who ordered more than once

In [0]:
df_multiple_order_user_metrics = df_agg_fact_order.join(df_multiple_order_users,
                                                        on='user_id',
                                                        how='inner')
print(df_multiple_order_user_metrics.count())

### Finding average gap

In [0]:
window1 = Window.partitionBy('user_id').orderBy('CREATION_TIME_UTC')
df_fact_order_prev = df_multiple_order_user_metrics.withColumn('prev_order_dt',F.lag('CREATION_TIME_UTC').over(window1))
df_fact_order_prev = df_fact_order_prev.withColumn('time_bw_order',F.datediff(F.col('CREATION_TIME_UTC'),F.col('prev_order_dt')))

In [0]:
df_fact_order_prev = df_fact_order_prev.filter(F.col('prev_order_dt').isNotNull())

In [0]:
#df_fact_order_prev.display()
agg_df_avg_gap = df_fact_order_prev.groupBy('user_id').agg(F.round(F.avg('time_bw_order'),3).alias('average_gap'))

In [0]:
agg_df_avg_gap = agg_df_avg_gap.withColumn('average_gap',
                                    F.col('average_gap').cast(StringType()))
agg_df_avg_gap.display()

In [0]:
df_avg_gap = agg_df_avg_gap.unionByName(df_onetime_user_gap)
print(agg_df_avg_gap.count())
print(df_avg_gap.count())
df_avg_gap.display()

### Finding average % change in spend

In [0]:
df_perct_change = df_multiple_order_user_metrics.filter(F.col('amount')!=0)

In [0]:
window1 = Window.partitionBy('user_id').orderBy('CREATION_TIME_UTC')
df_perct_change = df_perct_change.withColumn('prev_amt',F.lag('amount').over(window1))

In [0]:
df_perct_change = df_perct_change.withColumn('spend_change_pct',
              F.when(
            F.col("prev_amt") > 0,
            F.round(
                (
                    (F.col("amount") - F.col("prev_amt")) /
                    F.col("prev_amt")
                ) * 100,
                2
            )
        ).otherwise(F.lit(None).cast("double"))
)    

In [0]:
df_perct_change.display()

In [0]:
df_filtered_perct = df_perct_change.filter(F.col('spend_change_pct').isNotNull())
df_filtered_perct = df_filtered_perct.groupBy('user_id').agg(F.round(F.avg('spend_change_pct'),3).alias('avg_perct_chg'))

In [0]:
df_filtered_perct = df_filtered_perct.withColumn('avg_perct_chg',
                                F.col('avg_perct_chg').cast(StringType()))

In [0]:
df_filtered_perct.display()

In [0]:
df_avg_perct_change = df_filtered_perct.unionByName(df_onetime_user_perct)
print(df_filtered_perct.count())
print(df_avg_perct_change.count())
df_avg_perct_change.display()

### Days since last order

### Find the latest order w.r.t each user_id
### Find the diff between latest order and snapshot_date

In [0]:
snapshot_date = (df_fact_order.agg(F.max('CREATION_TIME_UTC').alias('snapshot_date')).first()['snapshot_date'])

In [0]:
df_fact_order_latest = df_fact_order.groupBy('user_id').agg(F.max('CREATION_TIME_UTC').alias('max_order_dt'))
df_fact_order_latest.display()

In [0]:
df_fact_order_latest = df_fact_order_latest.withColumn('days_since_last_order',
                                F.datediff(F.lit(snapshot_date),F.col('max_order_dt')))

df_fact_order_latest = df_fact_order_latest.drop('max_order_dt')
df_fact_order_latest.display()

In [0]:
print(df_fact_order_latest.count())

## Final Merge of all 3
df_fact_order_latest -- days_since_last_order.   
df_avg_perct_change --     
df_avg_gap.    

In [0]:
df_churn_indicators = (df_fact_order_latest
                .join(df_avg_perct_change,on='user_id',how='inner')
                .join(df_avg_gap,on='user_id',how='inner'))

In [0]:
df_churn_indicators = df_churn_indicators.withColumn('at_risk',
                               F.when(F.col('days_since_last_order')>45,True).otherwise(False))

In [0]:
print(df_churn_indicators.count())
df_churn_indicators.display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.mart.churn_indicators

In [0]:
df_churn_indicators.write.mode('append').saveAsTable('global_partner_project.mart.churn_indicators')